In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
import zipfile
import requests
import logging

# === Paths ===
BASE_DIR    = Path(r"D:\Shikha\bias_project")
GLOVE_DIR   = BASE_DIR / "models" / "glove"
OUTPUT_DIR  = BASE_DIR / "outputs" / "embeddings" / "glove"
WORD_LISTS  = BASE_DIR / "word_lists" / "caliskan_2017"

GLOVE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# === GloVe file — using GloVe 6B 100d (small, fast, good for demo) ===
GLOVE_URL  = "https://nlp.stanford.edu/data/glove.6B.zip"
GLOVE_ZIP  = GLOVE_DIR / "glove.6B.zip"
GLOVE_FILE = GLOVE_DIR / "glove.6B.100d.txt"

# === Download if not already present ===
if GLOVE_FILE.exists():
    print(f"GloVe file already exists: {GLOVE_FILE}")
else:
    print("Downloading GloVe 6B vectors (~822MB)...")
    r = requests.get(GLOVE_URL, stream=True)
    total = int(r.headers.get('content-length', 0))
    downloaded = 0
    with open(GLOVE_ZIP, 'wb') as f:
        for chunk in r.iter_content(chunk_size=1024*1024):
            f.write(chunk)
            downloaded += len(chunk)
            pct = downloaded / total * 100 if total else 0
            print(f"\r  Downloaded {downloaded/1e6:.1f}MB ({pct:.1f}%)", end='')
    print("\nExtracting...")
    with zipfile.ZipFile(GLOVE_ZIP, 'r') as z:
        z.extract("glove.6B.100d.txt", GLOVE_DIR)
    print(f"Done. Saved to {GLOVE_FILE}")

print(f"\nGloVe file exists: {GLOVE_FILE.exists()}")
print(f"File size        : {GLOVE_FILE.stat().st_size / 1e6:.1f} MB")

  Downloaded 862.2MB (100.0%)
Extracting...
Done. Saved to D:\Shikha\bias_project\models\glove\glove.6B.100d.txt

GloVe file exists: True
File size        : 347.1 MB


In [2]:
# Load GloVe vectors into a dictionary
print("Loading GloVe vectors...")

glove = {}
with open(GLOVE_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        parts = line.split()
        word = parts[0]
        vector = np.array(parts[1:], dtype=np.float32)
        glove[word] = vector

print(f"Loaded {len(glove):,} word vectors")
print(f"Vector dimension: {len(next(iter(glove.values())))}")

# === Load WEAT 6 word lists (same as Word2Vec notebook) ===
def load_word_list(filepath):
    with open(filepath, 'r') as f:
        return [w.strip().lower() for w in f.readlines() if w.strip()]

career_terms  = load_word_list(WORD_LISTS / "career_terms_weat6.txt")
family_terms  = load_word_list(WORD_LISTS / "family_terms_weat6.txt")
male_names    = load_word_list(WORD_LISTS / "male_names_weat6.txt")
female_names  = load_word_list(WORD_LISTS / "female_names_weat6.txt")

print(f"\nWEAT 6 word lists loaded:")
print(f"Career terms : {career_terms}")
print(f"Family terms : {family_terms}")
print(f"Male names   : {male_names}")
print(f"Female names : {female_names}")

# Verify all WEAT words are in GloVe vocabulary
print(f"\nVocabulary check:")
all_weat_words = career_terms + family_terms + male_names + female_names
missing = [w for w in all_weat_words if w not in glove]
print(f"WEAT words in GloVe : {len(all_weat_words) - len(missing)}/{len(all_weat_words)}")
if missing:
    print(f"Missing             : {missing}")
else:
    print("All WEAT words found in GloVe vocabulary!")

Loading GloVe vectors...
Loaded 400,000 word vectors
Vector dimension: 100

WEAT 6 word lists loaded:
Career terms : ['executive', 'management', 'professional', 'corporation', 'salary', 'office', 'business', 'career']
Family terms : ['home', 'parents', 'children', 'family', 'cousins', 'marriage', 'wedding', 'relatives']
Male names   : ['john', 'paul', 'mike', 'kevin', 'steve', 'greg', 'jeff', 'bill']
Female names : ['amy', 'joan', 'lisa', 'sarah', 'diana', 'kate', 'ann', 'donna']

Vocabulary check:
WEAT words in GloVe : 32/32
All WEAT words found in GloVe vocabulary!


In [3]:
# WEAT scoring functions using GloVe vectors

def cosine_sim(v1, v2):
    if np.linalg.norm(v1) == 0 or np.linalg.norm(v2) == 0:
        return 0.0
    return np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2))

def association(word, attr_A, attr_B, embeddings):
    if word not in embeddings:
        return None
    w_vec = embeddings[word]
    vecs_A = [embeddings[w] for w in attr_A if w in embeddings]
    vecs_B = [embeddings[w] for w in attr_B if w in embeddings]
    if not vecs_A or not vecs_B:
        return None
    sim_A = np.mean([cosine_sim(w_vec, v) for v in vecs_A])
    sim_B = np.mean([cosine_sim(w_vec, v) for v in vecs_B])
    return sim_A - sim_B

def weat6_score_glove(embeddings, male_names, female_names, 
                       career_terms, family_terms):
    male_assoc = [association(w, career_terms, family_terms, embeddings)
                  for w in male_names]
    female_assoc = [association(w, career_terms, family_terms, embeddings)
                    for w in female_names]
    male_assoc   = [x for x in male_assoc if x is not None]
    female_assoc = [x for x in female_assoc if x is not None]
    if not male_assoc or not female_assoc:
        return None
    return np.mean(male_assoc) - np.mean(female_assoc)

# === Option 1: Global GloVe score (pretrained, no fine-tuning) ===
# This gives one score using raw pretrained GloVe — 
# reflects general English language bias, not state-specific

global_score = weat6_score_glove(glove, male_names, female_names,
                                  career_terms, family_terms)
print(f"=== Global GloVe WEAT6 Score (pretrained) ===")
print(f"Score     : {global_score:.4f}")
print(f"Direction : {'male→career' if global_score > 0 else 'female→career'}")
print(f"\nThis reflects bias in general English (Wikipedia + news).")
print(f"Next: fine-tune GloVe per state using our corpus.")

=== Global GloVe WEAT6 Score (pretrained) ===
Score     : 0.2165
Direction : male→career

This reflects bias in general English (Wikipedia + news).
Next: fine-tune GloVe per state using our corpus.


In [5]:
from mittens import GloVe as MittensGloVe
from sklearn.feature_extraction.text import CountVectorizer
import warnings
warnings.filterwarnings('ignore')

# Load corpus - reuse same CSVs as Word2Vec notebook
CORPUS_DIR = BASE_DIR / "corpus"

print("Loading corpora...")
cr_df       = pd.read_csv(CORPUS_DIR / "congressional_record" / "2024-01" / "speeches.csv")
ca_df       = pd.read_csv(CORPUS_DIR / "chronicling_america" / "1920-1922" / "articles.csv")
gd_df       = pd.read_csv(CORPUS_DIR / "gdelt" / "2024-01" / "gdelt_articles_with_text.csv")
gd_df       = gd_df[gd_df['text'].notna() & (gd_df['text'] != '')]
rd_posts    = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "posts.csv")
rd_comments = pd.read_csv(CORPUS_DIR / "reddit" / "2020-01" / "comments.csv")
print("Done.")

# Build per-state text dictionary (same as Word2Vec notebook)
state_texts = {}  # state_abbr -> single combined text string

def add_text(state_abbr, text):
    if not isinstance(text, str) or len(text.strip()) < 50:
        return
    if state_abbr not in state_texts:
        state_texts[state_abbr] = []
    state_texts[state_abbr].append(text.strip())

EXCLUDE = {'GU', 'PR', 'VI', 'AS', 'MP'}

for _, row in cr_df.iterrows():
    add_text(str(row['state']), str(row.get('text', '')))
for _, row in ca_df.iterrows():
    add_text(str(row['state_abbr']), str(row.get('text', '')))
for _, row in gd_df.iterrows():
    add_text(str(row['state']), str(row.get('text', '')))
for _, row in rd_posts.iterrows():
    text = str(row.get('title', '')) + ' ' + str(row.get('selftext', ''))
    add_text(str(row['state_abbr']), text)
for _, row in rd_comments.iterrows():
    add_text(str(row['state_abbr']), str(row.get('body', '')))

# Combine into single string per state
state_corpus_text = {
    abbr: ' '.join(texts)
    for abbr, texts in state_texts.items()
    if abbr not in EXCLUDE
}

print(f"States with text: {len(state_corpus_text)}")
print(f"Sample state text length: {len(state_corpus_text.get('CA', ''))} chars")

Loading corpora...
Done.
States with text: 50
Sample state text length: 437180 chars


In [10]:
from mittens import Mittens
from sklearn.feature_extraction.text import CountVectorizer
import numpy as np

results_glove = []

print(f"{'State':<8} {'Text(K)':>8} {'WEAT6':>10} {'Direction'}")
print("-" * 50)

for state_abbr in sorted(state_corpus_text.keys()):
    text = state_corpus_text[state_abbr]

    try:
        # Build vocabulary from corpus
        vectorizer = CountVectorizer(
            min_df       = 1,
            max_features = 5000,
            ngram_range  = (1, 1)
        )
        X = vectorizer.fit_transform([text])
        vocab = list(vectorizer.get_feature_names_out())

        # Check WEAT words coverage
        weat_in_vocab = [w for w in WEAT_WORDS if w in vocab]
        if len(weat_in_vocab) < 10:
            print(f"{state_abbr:<8} {len(text)//1000:>8} {'NO SCORE':>10} insufficient WEAT vocab")
            continue

        # Build co-occurrence matrix
        cooc_matrix = (X.T @ X).toarray().astype(np.float32)
        np.fill_diagonal(cooc_matrix, 0)

        # Get pretrained GloVe vectors for this vocabulary
        pretrained = {w: glove[w] for w in vocab if w in glove}

        # Fine-tune with Mittens
        mittens_model = Mittens(n=100, max_iter=100)
        fine_tuned = mittens_model.fit(
            cooc_matrix,
            vocab   = vocab,
            initial_embedding_dict = pretrained
        )

        # Build embedding dict
        ft_embeddings = {w: fine_tuned[i] for i, w in enumerate(vocab)}

        # Compute WEAT6
        score = weat6_score_glove(
            ft_embeddings, male_names, female_names,
            career_terms, family_terms
        )

        if score is not None:
            direction = "male→career" if score > 0 else "female→career"
            print(f"{state_abbr:<8} {len(text)//1000:>8} {score:>10.4f} {direction}")
            results_glove.append({
                'state':       state_abbr,
                'text_chars':  len(text),
                'weat6_score': round(score, 4),
                'method':      'glove_finetuned',
                'period':      'mixed_demo'
            })
        else:
            print(f"{state_abbr:<8} {len(text)//1000:>8} {'NO SCORE':>10} insufficient WEAT vocab")

    except Exception as e:
        print(f"{state_abbr:<8} ERROR: {e}")

print(f"\nStates scored : {len(results_glove)}")

State     Text(K)      WEAT6 Direction
--------------------------------------------------


Iteration 100: error 50042.44070

AK            188     0.0735 male→career


Iteration 100: error 8657.99097

AL             63   NO SCORE insufficient WEAT vocab


...

ND            126     0.0612 male→career


Iteration 100: error 19639.1141

NE             81     0.0711 male→career
NH             78   NO SCORE insufficient WEAT vocab


Iteration 70: error 11689.00730

KeyboardInterrupt: 

In [11]:
from collections import Counter
import re

results_glove = []

def get_word_frequencies(text):
    """Get word frequency counts from text"""
    words = re.findall(r'\b[a-z]+\b', text.lower())
    return Counter(words)

def weighted_weat6_glove(word_freqs, glove, male_names, female_names,
                          career_terms, family_terms):
    """
    Compute WEAT6 using pretrained GloVe vectors weighted by
    corpus word frequency for state-specific scoring
    """
    def weighted_association(target_word, attr_A, attr_B):
        if target_word not in glove:
            return None
        t_vec = glove[target_word]
        
        # Weight attribute vectors by their frequency in state corpus
        def weighted_mean(attr_words):
            vecs, weights = [], []
            for w in attr_words:
                if w in glove:
                    freq = word_freqs.get(w, 1)
                    vecs.append(glove[w])
                    weights.append(freq)
            if not vecs:
                return None
            weights = np.array(weights, dtype=np.float32)
            weights = weights / weights.sum()
            weighted_vec = np.average(vecs, axis=0, weights=weights)
            return cosine_sim(t_vec, weighted_vec)
        
        sim_A = weighted_mean(attr_A)
        sim_B = weighted_mean(attr_B)
        if sim_A is None or sim_B is None:
            return None
        return sim_A - sim_B

    male_assoc   = [weighted_association(w, career_terms, family_terms)
                    for w in male_names]
    female_assoc = [weighted_association(w, career_terms, family_terms)
                    for w in female_names]
    male_assoc   = [x for x in male_assoc if x is not None]
    female_assoc = [x for x in female_assoc if x is not None]

    if not male_assoc or not female_assoc:
        return None
    return np.mean(male_assoc) - np.mean(female_assoc)

print(f"{'State':<8} {'Text(K)':>8} {'WEAT6':>10} {'Direction'}")
print("-" * 50)

for state_abbr in sorted(state_corpus_text.keys()):
    text = state_corpus_text[state_abbr]
    word_freqs = get_word_frequencies(text)

    score = weighted_weat6_glove(
        word_freqs, glove, male_names, female_names,
        career_terms, family_terms
    )

    if score is not None:
        direction = "male→career" if score > 0 else "female→career"
        print(f"{state_abbr:<8} {len(text)//1000:>8} {score:>10.4f} {direction}")
        results_glove.append({
            'state':       state_abbr,
            'text_chars':  len(text),
            'weat6_score': round(score, 4),
            'method':      'glove_weighted',
            'period':      'mixed_demo'
        })
    else:
        print(f"{state_abbr:<8} {len(text)//1000:>8} {'NO SCORE':>10}")

print(f"\nStates scored : {len(results_glove)}")

State     Text(K)      WEAT6 Direction
--------------------------------------------------
AK            188     0.1703 male→career
AL             63     0.2114 male→career
AR             70     0.1780 male→career
AZ             98     0.1305 male→career
CA            437     0.2200 male→career
CO             65     0.1321 male→career
CT             94     0.1466 male→career
DE             78     0.1446 male→career
FL            139     0.1166 male→career
GA            163     0.1636 male→career
HI            156     0.1649 male→career
IA            142     0.1638 male→career
ID             54     0.2299 male→career
IL            405     0.0933 male→career
IN            344     0.2216 male→career
KS             57     0.2118 male→career
KY             91     0.1607 male→career
LA            119     0.1852 male→career
MA            169     0.1483 male→career
MD            248     0.1558 male→career
ME            158     0.2018 male→career
MI             59     0.2103 male→career
MN      

In [12]:
# Save GloVe scores to CSV
scores_glove_df = pd.DataFrame(results_glove).sort_values('state')

OUTPUT_GLOVE = OUTPUT_DIR / "weat6_scores_glove.csv"
scores_glove_df.to_csv(OUTPUT_GLOVE, index=False)

print("=== WEAT6 Scores — GloVe Weighted (all 50 states) ===\n")
print(f"{'State':<8} {'WEAT6 Score':>12} {'Bias Direction'}")
print("-" * 40)
for _, row in scores_glove_df.iterrows():
    direction = "male→career" if row['weat6_score'] > 0 else "female→career"
    print(f"{row['state']:<8} {row['weat6_score']:>12.4f} {direction}")

print(f"\nAll 50 states scored")
print(f"Global GloVe baseline : 0.2165")
print(f"State range           : {scores_glove_df['weat6_score'].min():.4f} to {scores_glove_df['weat6_score'].max():.4f}")
print(f"Saved to              : {OUTPUT_GLOVE}")

=== WEAT6 Scores — GloVe Weighted (all 50 states) ===

State     WEAT6 Score Bias Direction
----------------------------------------
AK             0.1703 male→career
AL             0.2114 male→career
AR             0.1780 male→career
AZ             0.1305 male→career
CA             0.2200 male→career
CO             0.1321 male→career
CT             0.1466 male→career
DE             0.1446 male→career
FL             0.1166 male→career
GA             0.1636 male→career
HI             0.1649 male→career
IA             0.1638 male→career
ID             0.2299 male→career
IL             0.0933 male→career
IN             0.2216 male→career
KS             0.2118 male→career
KY             0.1607 male→career
LA             0.1852 male→career
MA             0.1483 male→career
MD             0.1558 male→career
ME             0.2018 male→career
MI             0.2103 male→career
MN             0.1366 male→career
MO             0.1646 male→career
MS             0.1689 male→career
MT             0.

In [8]:
# Quick check — what parameters does Mittens accept?
import inspect
from mittens import GloVe as MittensGloVe
print(inspect.signature(MittensGloVe.__init__))

(self, n=100, xmax=100, alpha=0.75, max_iter=100, learning_rate=0.05, tol=0.0001, display_progress=10, log_dir=None, log_subdir=None, test_mode=False)
